# Semana 14 Workshop: Primer Modelo de Machine Learning
## Dataset: UCDP — Conflictos Armados Globales

**Pregunta de investigación:** ¿Qué tan bien predice la frecuencia mensual de eventos violentos el número de muertes estimadas? ¿Qué otras variables del panel mensual mejoran esa predicción?

---

### Conexión con el proyecto
Ya calculamos en semanas anteriores que la **regresión lineal simple** (eventos → muertes) tiene un R² bajo. Ahora vamos más allá: un **Decision Tree** puede capturar relaciones no lineales y puede incluir más variables. Veremos si mejora la predicción y qué variables resultan más importantes.

---
## Setup

In [1]:
from pathlib import Path
import sys, site

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, accuracy_score
)

from ucdp_pipeline import (
    load_raw_events, clean_events,
    build_monthly_panel, build_global_monthly
)

sns.set_theme(style='whitegrid')
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = PROJECT_ROOT / 'data'
print('Setup listo!')

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Cargar y limpiar datos (mismo flujo de semanas anteriores)
raw = load_raw_events(DATA_DIR)
clean = clean_events(raw)
monthly_panel = build_monthly_panel(clean, min_year=2022)
global_monthly = build_global_monthly(monthly_panel)

print(f'Panel mensual global: {len(global_monthly)} meses')
print(f'Columnas: {global_monthly.columns.tolist()}')
global_monthly.head()

---
# Parte 1: Preparación de Datos

## 1.1 Identificar columnas numéricas del panel

In [ ]:
numeric_cols = global_monthly.select_dtypes(include=[np.number]).columns.tolist()

print(f'Columnas numéricas disponibles ({len(numeric_cols)}):')
for col in numeric_cols:
    print(f'  {col}: min={global_monthly[col].min():.1f}, max={global_monthly[col].max():.1f}, nulos={global_monthly[col].isnull().sum()}')

# EXPLICACIÓN:
# El panel global tiene una fila por mes (desde 2022).
# Cada fila resume: cuántos eventos hubo, cuántos muertos, cuántos civiles, etc.
# Con estas columnas vamos a construir el modelo de ML.

## 1.2 Seleccionar variable objetivo (target)

In [ ]:
# Target: muertes estimadas mensuales (fatalities_best)
target_col = 'fatalities_best'

print(f'Variable objetivo: {target_col}')
print(global_monthly[target_col].describe())

# Visualizar distribución
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(global_monthly[target_col], bins=20, color='#246A73', edgecolor='white')
axes[0].set_xlabel('Muertes estimadas por mes')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de muertes mensuales')

axes[1].hist(np.log1p(global_monthly[target_col]), bins=20, color='#D9A441', edgecolor='white')
axes[1].set_xlabel('log(1 + muertes)')
axes[1].set_title('Distribución en escala logarítmica')

plt.tight_layout()
plt.show()

# EXPLICACIÓN:
# fatalities_best es un número continuo → problema de REGRESIÓN
# La distribución es asimétrica (algunos meses tienen miles de muertes, otros pocas)
# Es exactamente la misma variable que analizamos con regresión lineal antes
# Ahora veremos si un Decision Tree lo predice mejor

## 1.3 Seleccionar features

In [ ]:
# Features para predecir las muertes mensuales
# Usamos variables que el modelo podría conocer *antes* de contar los muertos
feature_cols = [
    'events',              # Número de eventos violentos ese mes
    'civilian_fatalities', # Muertes civiles (ya es parte del total, pero es una variable)
]

# Agregar columnas extra si existen en el panel
optional_cols = ['uncertainty_total', 'high_fatalities', 'low_fatalities']
for col in optional_cols:
    if col in global_monthly.columns:
        feature_cols.append(col)
        print(f'  + {col} incluido')

# Si el panel solo tiene events y civilian_fatalities, está bien — eso es suficiente
feature_cols = [col for col in feature_cols if col in global_monthly.columns]

print(f'\nFeatures finales: {feature_cols}')

# EXPLICACIÓN DE LA SELECCIÓN:
# events: el predictor principal — más eventos ¿más muertes?
# civilian_fatalities: las muertes civiles son un subconjunto de fatalities_best
# uncertainty_total / high_fatalities / low_fatalities: capturan qué tan inciertos
#   son los datos ese mes (datos Candidate vs datos verificados)

# IMPORTANTE: NO incluimos fatalities_best en las features porque eso sería trampa
# (predecir algo usando el valor exacto que queremos predecir)

In [ ]:
# Limpiar: eliminar filas con nulos en las columnas seleccionadas
df_ml = global_monthly[feature_cols + [target_col]].dropna().copy()

X = df_ml[feature_cols]
y = df_ml[target_col]

print(f'Filas antes de limpiar: {len(global_monthly)}')
print(f'Filas usables para ML: {len(df_ml)}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

## 1.4 Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Meses de entrenamiento: {X_train.shape[0]}')
print(f'Meses de prueba:        {X_test.shape[0]}')

# NOTA IMPORTANTE para tu proyecto:
# Con datos de series de tiempo idealmente haríamos una división temporal
# (entrenar con 2022-2024, predecir 2025-2026). Aquí usamos split aleatorio
# por simplicidad del workshop. Eso es algo que puedes mencionar en la reflexión
# como limitación y posible mejora.

---
# Parte 2: Decision Tree Regressor

Patrón sklearn: Crear → Entrenar → Predecir → Evaluar

## 2.1 Entrenar el modelo

In [ ]:
# PASO 1: Crear el modelo
model = DecisionTreeRegressor(max_depth=5, random_state=42)

# PASO 2: Entrenar con los datos de entrenamiento
model.fit(X_train, y_train)

# PASO 3: Predecir en datos de prueba (que el modelo NUNCA vio)
y_pred = model.predict(X_test)

print('Modelo entrenado!')
print()

# Ver algunas predicciones vs realidad
comparacion = pd.DataFrame({
    'Mes': df_ml.index[y_test.index] if hasattr(df_ml.index, '__len__') else range(len(y_test)),
    'Muertes reales': y_test.values[:8].round(0).astype(int),
    'Muertes predichas': y_pred[:8].round(0).astype(int),
    'Error': (y_pred[:8] - y_test.values[:8]).round(0).astype(int)
})
print(comparacion.to_string(index=False))

# EXPLICACIÓN:
# El Decision Tree aprendió reglas del tipo:
# '¿events > 3000 ese mes? Si sí → más de X muertes esperadas'
# max_depth=5 limita la complejidad del árbol

## 2.2 Evaluar el modelo

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('=== DECISION TREE REGRESSOR (max_depth=5) ===')
print(f'RMSE: {rmse:,.0f}  muertes de error promedio por mes')
print(f'MAE:  {mae:,.0f}  muertes de error absoluto promedio')
print(f'R²:   {r2:.4f}  ({r2*100:.1f}% de la varianza explicada)')
print()

# COMPARACIÓN CON REGRESIÓN LINEAL DEL PROYECTO:
# En el notebook principal ya teníamos un R² bajo con regresión lineal
# El Decision Tree puede capturar relaciones no lineales
# Si R²_DT > R²_RegLineal: el árbol mejora la predicción
# Esto es un hallazgo valioso para defender en la presentación

rango = y.max() - y.min()
print(f'Contexto: el rango de muertes mensuales es {rango:,.0f}')
print(f'RMSE representa el {rmse/rango*100:.1f}% del rango total')

## 2.3 Detectar Overfitting

In [ ]:
r2_train = r2_score(y_train, model.predict(X_train))

print(f'R² entrenamiento: {r2_train:.4f}')
print(f'R² prueba:        {r2:.4f}')
print(f'Brecha (gap):     {r2_train - r2:.4f}')

if r2_train - r2 > 0.1:
    print('\n⚠️  Posible overfitting. El modelo memoriza los meses de entrenamiento.')
else:
    print('\n✅ Sin overfitting significativo.')

# CONTEXTO PARA TU PROYECTO:
# Con pocos meses (el panel empieza en 2022 → ~40 meses hasta 2026)
# el riesgo de overfitting es real. Un árbol sin límite de profundidad
# puede 'memorizar' los picos de muertes (e.g. el pico de Gaza en 2023)
# sin aprender un patrón generalizable.

## 2.4 Visualización: Real vs Predicho

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(y_test, y_pred, alpha=0.7, color='#246A73', s=80, zorder=3)
lim = max(y_test.max(), y_pred.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', linewidth=2, label='Predicción perfecta', zorder=2)

ax.set_xlabel('Muertes reales por mes', fontsize=12)
ax.set_ylabel('Muertes predichas por mes', fontsize=12)
ax.set_title(f'Decision Tree: Real vs Predicho\nR² = {r2:.4f} | RMSE = {rmse:,.0f} muertes/mes', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

# CÓMO INTERPRETAR:
# Puntos sobre la línea roja = predicción perfecta
# Puntos muy alejados de la línea = meses atípicos (picos como Gaza oct 2023, Ucrania)
# La forma en 'escalera' es característica de árboles de decisión
#  (predice el promedio de un grupo, no un valor continuo)

---
# Parte 3: Experimentar con max_depth

¿Qué profundidad minimiza el overfitting sin sacrificar demasiado poder predictivo?

In [ ]:
depths = [2, 3, 5, 7, 10, 15, 20, None]
results = []

for depth in depths:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)

    r2_tr = r2_score(y_train, dt.predict(X_train))
    r2_te = r2_score(y_test,  dt.predict(X_test))
    rmse_te = np.sqrt(mean_squared_error(y_test, dt.predict(X_test)))

    results.append({
        'max_depth': str(depth) if depth else 'None',
        'R2_train': round(r2_tr, 4),
        'R2_test': round(r2_te, 4),
        'gap': round(r2_tr - r2_te, 4),
        'RMSE_test': round(rmse_te, 0)
    })

results_df = pd.DataFrame(results)
print('=== COMPARACIÓN POR PROFUNDIDAD ===')
print(results_df.to_string(index=False))

mejor = results_df.loc[results_df['R2_test'].idxmax()]
print(f'\n✅ Mejor max_depth: {mejor["max_depth"]} — R²_test={mejor["R2_test"]}, RMSE={mejor["RMSE_test"]:,.0f}')

In [ ]:
# Curva de overfitting
fig, ax = plt.subplots(figsize=(10, 6))

x_labels = results_df['max_depth'].tolist()
x_pos = list(range(len(x_labels)))

ax.plot(x_pos, results_df['R2_train'], 'o-', color='steelblue',
        linewidth=2, markersize=8, label='R² Entrenamiento')
ax.plot(x_pos, results_df['R2_test'],  's-', color='#D9A441',
        linewidth=2, markersize=8, label='R² Prueba')
ax.fill_between(x_pos, results_df['R2_train'], results_df['R2_test'],
                alpha=0.15, color='red', label='Brecha (overfitting)')

ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels)
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_title('Curva de Overfitting — Dataset UCDP\nPredecir muertes mensuales con Decision Tree', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('Observaciones:')
print('- Con pocas filas (~40 meses), el riesgo de overfitting es alto')
print('- max_depth=None: el árbol memoriza cada mes atípico (Gaza, Ucrania...)')
print('- El punto óptimo: mayor R²_test con gap < 0.10')

### Análisis de profundidad

Con un dataset de ~40 meses, la profundidad óptima suele ser baja (2–4). Con árboles muy profundos el modelo 'recuerda' los meses de picos extremos (ejemplo: un mes con 20,000 muertes en Ucrania) sin aprender un patrón real.

Esto tiene **sentido de dominio**: los conflictos armados tienen dinámicas no lineales — un escalamiento repentino no se predice solo con el número de eventos del mes anterior.

---
# Parte 4: Decision Tree Classifier (Extensión Guiada)

En vez de predecir el número exacto de muertes, clasificamos cada mes como de **Baja, Media o Alta** letalidad.

In [ ]:
# Crear 3 categorías de letalidad mensual
def categorize(value, thresholds):
    if value < thresholds[0]:
        return 'Baja'
    elif value < thresholds[1]:
        return 'Media'
    else:
        return 'Alta'

thresholds = [y.quantile(0.33), y.quantile(0.66)]
print(f'Umbrales de letalidad mensual:')
print(f'  Baja  : < {thresholds[0]:,.0f} muertes/mes')
print(f'  Media : {thresholds[0]:,.0f} – {thresholds[1]:,.0f} muertes/mes')
print(f'  Alta  : > {thresholds[1]:,.0f} muertes/mes')

y_cat = y.apply(lambda x: categorize(x, thresholds))
print(f'\nDistribución:')
print(y_cat.value_counts())

# APLICACIÓN PRÁCTICA:
# Este clasificador podría usarse como sistema de alerta temprana:
# Si el modelo predice 'Alta' para el mes siguiente, los organismos
# humanitarios pueden movilizar recursos preventivamente.

In [ ]:
# División para clasificación
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cat, test_size=0.2, random_state=42
)

# Entrenar clasificador
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train_c, y_train_c)

# Predecir y evaluar
y_pred_c = clf.predict(X_test_c)
accuracy = accuracy_score(y_test_c, y_pred_c)

print(f'Accuracy del clasificador: {accuracy:.2%}')
print(f'(De cada 10 meses, clasifica bien ~{accuracy*10:.0f})')

# Overfitting en clasificación
train_acc = accuracy_score(y_train_c, clf.predict(X_train_c))
print(f'\nAccuracy entrenamiento: {train_acc:.2%}')
print(f'Accuracy prueba:        {accuracy:.2%}')
print(f'Gap:                    {train_acc - accuracy:.2%}')

if train_acc - accuracy > 0.10:
    print('\n⚠️  Overfitting en el clasificador.')
else:
    print('\n✅ Clasificador sin overfitting significativo.')

---
# Parte 5: Interpretación y Reflexión

In [ ]:
# Feature importance: ¿qué variable importa más para predecir muertes?
best_depth = 5   # ← ACTUALIZA con el mejor max_depth de la Parte 3
best_model = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_model.fit(X_train, y_train)

importance = pd.DataFrame({
    'variable': feature_cols,
    'importancia': best_model.feature_importances_
}).sort_values('importancia', ascending=False)

print('Importancia de variables para predecir muertes mensuales:')
print(importance.to_string(index=False))
print()
print(f'Variable más importante: {importance.iloc[0]["variable"]} ({importance.iloc[0]["importancia"]*100:.1f}%)')

In [ ]:
# Visualizar feature importance
fig, ax = plt.subplots(figsize=(8, max(4, len(feature_cols) * 0.8)))

colores = ['#246A73' if i == 0 else '#9dbfc4' for i in range(len(importance))]
ax.barh(importance['variable'], importance['importancia'], color=colores)
ax.set_xlabel('Importancia relativa', fontsize=12)
ax.set_title(
    f'Decision Tree (max_depth={best_depth})\nVariables más importantes para predecir letalidad mensual UCDP',
    fontsize=12
)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# INTERPRETACIÓN PARA TU PROYECTO:
# Si 'civilian_fatalities' tiene alta importancia: las muertes civiles
#   son buen predictor de la letalidad total — algunos conflictos son
#   más dañinos para civiles que otros (Gaza vs Ucrania por ejemplo)
# Si 'events' tiene baja importancia: confirma la hipótesis de tu proyecto
#   de que frecuencia de eventos NO explica bien la letalidad

In [ ]:
# Comparar Decision Tree vs Regresión Lineal del proyecto
# (usa el R² que obtuviste en el notebook principal)

r2_final = r2_score(y_test, best_model.predict(X_test))
rmse_final = np.sqrt(mean_squared_error(y_test, best_model.predict(X_test)))

print('=== RESUMEN COMPARATIVO ===')
print(f'Decision Tree (max_depth={best_depth}):')
print(f'  R²   = {r2_final:.4f}')
print(f'  RMSE = {rmse_final:,.0f} muertes/mes')
print()
print('Regresión Lineal Simple (del proyecto):')
print('  R²   = [ver regression_metrics del notebook principal]')
print('  RMSE = [ver regression_metrics del notebook principal]')
print()
print('Interpretación:')
print('Si R²_DT > R²_RegLineal: el árbol captura relaciones no lineales')
print('Si son similares: la relación eventos-muertes es fundamentalmente lineal o débil')
print('De cualquier forma, un R² bajo confirma la hipótesis del proyecto:')
print('  → la frecuencia de eventos por sí sola no explica la letalidad')

## Reflexión — Preguntas del workshop

### 1. Mejor max_depth

Con solo ~40 meses de datos (2022–2026), el mejor `max_depth` probablemente es bajo (2–4). Árboles profundos memorizan los meses atípicos como los picos de Gaza o Ucrania, sin aprender un patrón generalizable. Esto es una limitación inherente al tamaño del panel mensual.

### 2. Calidad del modelo

Un R² moderado o bajo no es una falla del trabajo — es un **hallazgo valioso**. Significa que la frecuencia mensual de eventos no es suficiente para predecir las muertes. Eso es consistente con la hipótesis central del proyecto: algunos conflictos concentran muchas muertes en pocos episodios de alta intensidad, no en muchos eventos de baja letalidad.

### 3. Importancia de variables

Si `civilian_fatalities` domina la importancia: indica que el tipo de violencia (ataques a civiles vs combates entre fuerzas) es el mejor predictor de la letalidad total. Esto apoya la comparación entre conflictos (Gaza tiene alta proporción civil, Ucrania tiene más combates militares).

### 4. Próximos pasos

- Usar el panel **por conflicto** en vez del global (más filas, más variabilidad)
- Incluir variables del tipo de violencia como feature (State-based, Non-state, One-sided)
- Probar Random Forest para suavizar el overfitting
- División temporal: entrenar 2022–2024, predecir 2025 (más realista para series de tiempo)

---

## Checklist de entrega

- [ ] Todas las celdas ejecutadas (Kernel > Restart & Run All)
- [ ] Parte 1: Panel mensual UCDP preparado
- [ ] Parte 2: Decision Tree entrenado y evaluado con RMSE y R²
- [ ] Parte 3: Curva de overfitting graficada, mejor depth identificado
- [ ] Parte 4: Clasificador Baja/Media/Alta letalidad entrenado
- [ ] Parte 5: Feature importance visualizada y reflexión escrita

---

### Conexión con Milestone 3

Este notebook es el componente de ML del Milestone 3. Lo que demuestras aquí:

| Habilidad del workshop | Aplicación en el proyecto |
|---|---|
| Decision Tree Regressor | Predecir muertes mensuales (supera la regresión lineal simple) |
| Análisis de overfitting | Discutir límites del modelo por tamaño de datos |
| Feature importance | Qué variables del UCDP realmente explican la letalidad |
| Clasificador Baja/Media/Alta | Potencial sistema de alerta temprana humanitaria |

---
*Semana 14 Workshop — Analítica de Datos — Universidad Cooperativa de Colombia*